In [1]:
import matplotlib
import os
import sys
sys.path.append('..')
from er_simulator.wc_model import WCTaskSim
from er_simulator.functions import resample_signal
from er_simulator.synaptic_weights_matrices import normalize, generate_synaptic_weights_matrices
from er_simulator.read_utils import generate_sw_matrices_from_mat
from er_simulator.load_wc_params import load_wc_params
from er_simulator import functions
from er_simulator.boldIntegration import BWBoldModel
import numpy as np
from tqdm import tqdm 
from tqdm import trange
import matplotlib.pyplot as plt
from scipy import signal, stats, io
from scipy.io import savemat
from er_simulator.read_utils import get_project_root


matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['font.size'] = 10
plt.rcParams['image.cmap'] = 'plasma'
np.set_printoptions(suppress=True)
%load_ext autoreload
%autoreload 2

In [3]:
N = 100 # Sample size
microtime = 2/16
design = "02_EVENT_[2s_TR]_[1s_DUR]_[6s_ISI]_[100_TRIALS]_[COACT]";
HRF_type =  "[VarHRF]" # "[FixedHRF]"

results_folder = f'{design}_{HRF_type}'
mat_path = os.path.join('..', '02_designs', f'{design}.mat')
config_file = os.path.join('..', '01_configs', f'{results_folder}.yaml')

In [ ]:
for i in trange(N):
    
    # Create subject folder
    os.makedirs(os.path.join('..', '03_results', results_folder, fr'{i+1:03d}_Subject'), exist_ok=True)
    
    # Load simulation parameters from config file
    wc_sim = WCTaskSim.from_config(config_file)

    # Save HRF parameters
    wc_sim.boldModel.save_imp_with_params(os.path.join(fr'..', 
                                                       '03_results',
                                                       results_folder, 
                                                       fr'{i+1:03d}_Subject',
                                                       'HRF_params.mat'), s_type='mat')

    # Generate task time-series in 5 ms resolution
    wc_sim.generate_full_series(compute_bold=True)
    output_task = wc_sim.output.copy()

    # Calculate task-state envelopes
    task_env_dict = wc_sim.compute_envelope_and_resample(wc_sim.output['syn_act'], srate=wc_sim.mTime, microtime=microtime)

    # Downsample task BOLD to Micro-Time (MT) resolution = TR/16  
    task_BOLD_oscill_MT,_ = resample_signal(output_task['mtime'], output_task["BOLD"], wc_sim.mTime, microtime)

    # Generate co-activations in 10 ms resolution
    time_coact,_,task_BOLD_coact = wc_sim.generate_coactivation_by_mat(wc_sim.mat_path, dt=None, normalize_constant=1)

    # Downsample co-activations to Micro-Time (MT) resolution = TR/16  
    task_BOLD_coact_MT,_ =  resample_signal(time_coact, task_BOLD_coact,  wc_sim.mTime, microtime)
    
    # Generate rest time-series in 5 ms resolution
    output_rest = wc_sim.generate_rest_series(compute_bold=True)

    # Calculate resting-state envelopes
    rest_env_dict = wc_sim.compute_envelope_and_resample(output_rest['syn_act'], srate=wc_sim.mTime, microtime=microtime)

    # Downsample rest BOLD to Micro-Time (MT) resolution = TR/16
    rest_BOLD_oscill_MT,_ = resample_signal(output_rest['mtime'], output_rest["BOLD"], wc_sim.mTime, microtime)

    # Save time-series
    savemat(os.path.join(fr'..',
                         '03_results',
                         results_folder,
                         fr'{i+1:03d}_Subject',
                         'time_series.mat'),
                          {**task_env_dict, **rest_env_dict,
                           'task_BOLD_oscill_MT': task_BOLD_oscill_MT,
                           'task_BOLD_coact_MT':  task_BOLD_coact_MT,
                           'rest_BOLD_oscill_MT': rest_BOLD_oscill_MT}) 
    
    # Clear
    del output_task, task_BOLD_oscill_MT, task_BOLD_coact_MT, output_rest, wc_sim 

In [7]:
i = 0

# Create subject folder
os.makedirs(os.path.join('..', '03_results', results_folder, fr'{i+1:03d}_Subject'), exist_ok=True)

# Load simulation parameters from config file
wc_sim = WCTaskSim.from_config(config_file)

# Save HRF parameters
wc_sim.boldModel.save_imp_with_params(os.path.join(fr'..', 
                                                   '03_results',
                                                   results_folder, 
                                                   fr'{i+1:03d}_Subject',
                                                   'HRF_params.mat'), s_type='mat')

# Generate task time-series in 5 ms resolution
wc_sim.generate_full_series(compute_bold=True)
output_task = wc_sim.output.copy()

# Calculate task-state envelopes
task_env_dict = wc_sim.compute_envelope_and_resample(wc_sim.output['syn_act'], srate=wc_sim.mTime, microtime=microtime)

# Downsample task BOLD to Micro-Time (MT) resolution = TR/16  
task_BOLD_oscill_MT,_ = resample_signal(output_task['mtime'], output_task["BOLD"], wc_sim.mTime, microtime)

# Generate co-activations in 10 ms resolution
time_coact,_,task_BOLD_coact = wc_sim.generate_coactivation_by_mat(wc_sim.mat_path, dt=None, normalize_constant=1)

# Downsample co-activations to Micro-Time (MT) resolution = TR/16  
task_BOLD_coact_MT,_ =  resample_signal(time_coact, task_BOLD_coact,  wc_sim.mTime, microtime)

# Generate rest time-series in 5 ms resolution
output_rest = wc_sim.generate_rest_series(compute_bold=True)

# Calculate resting-state envelopes
rest_env_dict = wc_sim.compute_envelope_and_resample(output_rest['syn_act'], srate=wc_sim.mTime, microtime=microtime)

# Downsample rest BOLD to Micro-Time (MT) resolution = TR/16
rest_BOLD_oscill_MT,_ = resample_signal(output_rest['mtime'], output_rest["BOLD"], wc_sim.mTime, microtime)

# Save time-series
savemat(os.path.join(fr'..',
                     '03_results',
                     results_folder,
                     fr'{i+1:03d}_Subject',
                     'time_series.mat'),
                      {**task_env_dict, **rest_env_dict,
                       'task_BOLD_oscill_MT': task_BOLD_oscill_MT,
                       'task_BOLD_coact_MT':  task_BOLD_coact_MT,
                       'rest_BOLD_oscill_MT': rest_BOLD_oscill_MT}) 

# Clear
# del task_env_dict, rest_env_dict, task_BOLD_oscill_MT, task_BOLD_coact_MT, rest_BOLD_oscill_MT, output_task, output_rest, wc_sim, time_coact

In [8]:
new_rest_keys = {f"rest_{key}": value for key, value in rest_env_dict.items()}
rest_env_dict.update(new_rest_keys)

new_task_keys = {f"task_{key}": value for key, value in task_env_dict.items()}
task_env_dict.update(new_task_keys)

savemat(os.path.join(fr'..',
                     '03_results',
                     results_folder,
                     fr'{i+1:03d}_Subject',
                     'time_series.mat'),
                      {**task_env_dict, **rest_env_dict,
                       'task_BOLD_oscill_MT': task_BOLD_oscill_MT,
                       'task_BOLD_coact_MT':  task_BOLD_coact_MT,
                       'rest_BOLD_oscill_MT': rest_BOLD_oscill_MT}) 

In [10]:
print(rest_env_dict.keys())

dict_keys(['e_hil_d2', 'e_hil_d5', 'e_peak_100', 'e_peak_450', 'e_peak_600', 'e_peak_1000', 'rest_e_hil_d2', 'rest_e_hil_d5', 'rest_e_peak_100', 'rest_e_peak_450', 'rest_e_peak_600', 'rest_e_peak_1000'])
